# 🎙️ Wav2Vec2 Embedding Extractor — Laptop Version

This notebook extracts 768-dimensional embeddings from raw audio (`.flac` / `.wav`) using **facebook/wav2vec2-base** directly on your **laptop**.

### Features:
- Uses local ASVspoof 2019 dataset from `C:\Users\rahul\Downloads\AVSspoof_LA\LA`
- Auto-detects local GPU or configures multi-threaded CPU inference
- Batching & progress bar via `tqdm`
- Outputs saved directly to `data/embeddings/`

--- 
## 1. Setup & Environment

In [2]:
import os
import sys
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
import torchaudio
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
from tqdm.auto import tqdm

# Project Root
PROJECT_ROOT = Path("..").resolve().parent if Path("..").resolve().name == "notebooks" else Path(".").resolve()
DATA_ROOT = Path(r"C:\Users\rahul\Downloads\AVSspoof_LA\LA")
OUT_DIR = PROJECT_ROOT / "data" / "embeddings"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count() - 1))
    print(f"Using Laptop CPU with {torch.get_num_threads()} threads")
else:
    print(f"Using Local GPU: {torch.cuda.get_device_name(0)}")

c:\Deepfake-voice-detection\miniproject\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Laptop CPU with 15 threads


--- 
## 2. Load Pretrained Wav2Vec2 Model

In [3]:
MODEL_NAME = "facebook/wav2vec2-base"
print(f"Loading {MODEL_NAME}...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Model loaded successfully!")

Loading facebook/wav2vec2-base...


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 12266.77it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!


--- 
## 3. Audio Loading & Extraction Routine

In [4]:
TARGET_SR = 16000

def extract_clip_embedding(audio_path):
    # Use soundfile for cross-platform, robust audio loading (avoids TorchCodec dependency on Windows)
    try:
        data, sr = sf.read(str(audio_path))
        if data.ndim > 1:
            data = np.mean(data, axis=1)  # downmix stereo to mono
        if sr != TARGET_SR:
            waveform = torch.from_numpy(data).float().unsqueeze(0)
            waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)
            data = waveform.squeeze(0).numpy()
    except Exception:
        waveform, sr = torchaudio.load(str(audio_path))
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != TARGET_SR:
            waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)
        data = waveform.squeeze(0).numpy()
    
    inputs = feature_extractor(data, sampling_rate=TARGET_SR, return_tensors="pt")
    input_values = inputs.input_values.to(device)
    
    with torch.no_grad():
        outputs = model(input_values)
        # Mean pool across time dimension
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0).cpu()
    return embedding

--- 
## 4. Run Extraction on Split (Example: Dev Subset or Custom Audio)

In [5]:
# Example on a sample of clips or custom user recordings
protocol_file = DATA_ROOT / "ASVspoof2019_LA_cm_protocols" / "ASVspoof2019.LA.cm.dev.trl.txt"
flac_dir = DATA_ROOT / "ASVspoof2019_LA_dev" / "flac"

LABEL_MAP = {"bonafide": 0, "spoof": 1}

# Read first 50 clips as demonstration
LIMIT = 50
entries = []
with open(protocol_file, "r") as f:
    for line in f:
        p = line.strip().split()
        if len(p) >= 5:
            entries.append((p[1], LABEL_MAP[p[4].lower()]))
        if len(entries) >= LIMIT:
            break

embeddings, labels, filenames = [], [], []
print(f"Extracting embeddings for {len(entries)} clips...")

for fn, label in tqdm(entries):
    audio_path = flac_dir / f"{fn}.flac"
    if audio_path.exists():
        emb = extract_clip_embedding(audio_path)
        embeddings.append(emb)
        labels.append(label)
        filenames.append(fn)

embeddings_tensor = torch.stack(embeddings)
labels_tensor = torch.tensor(labels)
print(f"Done! Output shape: {embeddings_tensor.shape}")

Extracting embeddings for 50 clips...


100%|██████████| 50/50 [00:31<00:00,  1.60it/s]

Done! Output shape: torch.Size([50, 768])
